# County HSDS download and BA weather aggregation

This notebook stages weather at reviewed county grid points, then uses population weights to form BA weather. It follows the method demonstrated in the [county point-selection notebook](county_weather_point_selection.ipynb). The default MISO example stages all counties but only the first hour of 2023.

**Inputs:** The supplied national county-point mapping, BA service territories, fixed 2020 county population, and matching local BC-HRRR/NSRDB H5 files or access to HSDS for missing files.

**Requirements:** The repository environment and matching local county-weather H5 files, or a local HSDS service at `http://localhost:5101` with access to `nrel-pds-hsds` to download missing files.

**Outputs:** Reusable county-weather H5 files and `MISO_2023_one_hour_ba_weather.csv` under `../../data_outputs/data_flow/county_hsds_download_and_ba_weather_aggregation/`. The CSV has TELL's seven-column weather schema; the [load-forecast notebook](tell_load_forecast_data_flow.ipynb) uses a separately supplied complete historical archive.

Run the code cells from top to bottom. See the [setup instructions](../../README.md#quick-start).


## Setup

The cache checks in Step 2 decide whether a weather download is needed.


In [1]:
from pathlib import Path

from IPython.display import display
import h5py
import numpy as np
import pandas as pd
from rex import Resource

OUTPUT_DIR = Path("../../data_outputs/data_flow/county_hsds_download_and_ba_weather_aggregation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Step 1: Read reviewed county grid points

`county_centroid_regrid.csv` is the supplied national crosswalk. The preceding notebook demonstrates its selection method using Arthur County; it does not regenerate this national input.

In [2]:
WEATHER_SOURCES = ["bchrrr", "nsrdb"]

# Read county_fips as text so leading zeros can be restored/preserved.
regrid_matches = pd.read_csv(
    "../../data_inputs/county_weather/county_centroid_regrid.csv",
    dtype={"county_fips": str},
)
regrid_matches["county_fips"] = regrid_matches["county_fips"].str.zfill(5)
selected_source_matches = regrid_matches[
    regrid_matches["weather_source"].isin(WEATHER_SOURCES)
].copy()
display(selected_source_matches.head())

,county_fips,county_name,state_name,weather_source,county_pop_lat,county_pop_lon,selected_gid,grid_lat,grid_lon,distance_km
1,01001,Autauga County,Alabama,bchrrr,32.500197,-86.487818,1827777,32.498220,-86.487335,0.224451
2,01001,Autauga County,Alabama,nsrdb,32.500197,-86.487818,926773,32.490000,-86.500000,1.609640
4,01003,Baldwin County,Alabama,bchrrr,30.537375,-87.761515,1772629,30.543240,-87.768130,0.909203
5,01003,Baldwin County,Alabama,nsrdb,30.537375,-87.761515,900257,30.530000,-87.780000,1.951121
7,01005,Barbour County,Alabama,bchrrr,31.844091,-85.301177,1891804,31.841358,-85.310640,0.944123


## Step 2: Stage 2023 county weather

Check local weather files, then download only those that are missing. With `HOUR_LIMIT = 1`, this example stages the first hourly timestamp of 2023 at the selected county grid points.

In [3]:
HOUR_LIMIT = 1  # Stage only the first top-of-hour timestamp so the worked example stays small.

COUNTY_H5_DIR = OUTPUT_DIR / "county_weather_h5"
COUNTY_H5_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_RESOURCE_PATHS = {
    "bchrrr": "/nrel/wtk/bchrrr/v1.0.0/bchrrr_conus_2023.h5",
    "nsrdb": "/nrel/nsrdb/GOES/aggregated/v4.0.0/nsrdb_2023.h5",
}

SOURCE_VARIABLES = {
    "bchrrr": [
        "temperature_2m",
        "windspeed_10m",
        "relativehumidity_2m",
        "pressure_0m",
        "specifichumidity_2m",
    ],
    "nsrdb": ["ghi"],
}

NATIVE_WEATHER_COLUMNS = [
    "temperature_2m",
    "specifichumidity_2m",
    "windspeed_10m",
    "ghi",
    "relativehumidity_2m",
    "pressure_0m",
]

county_h5_by_source = {}
for weather_source in WEATHER_SOURCES:
    filename = f"{weather_source}_2023_all_county_weather.h5"
    county_h5_by_source[weather_source] = COUNTY_H5_DIR / filename

### Check existing weather files

This cell checks each local H5 file before deciding whether HSDS is needed. It reads metadata and identifiers without loading weather-value arrays; any downloads happen in the next code cell.

- **Missing file:** queue it for the next download cell.
- **Matching file:** reuse it after all checks pass.
- **Validation fails:** stop with an error and leave the existing file unchanged.

Reuse requires matching county FIPS and grid IDs in the same order, required variable shapes, and exact hourly UTC timestamps. With the default `HOUR_LIMIT = 1`, the expected timestamp is `2023-01-01 00:00 UTC`.

**County files:** all three source metadata attributes (`kind`, `source`, and `hsds_resource_path`) are required and must match.

In [4]:
sources_to_download = []  # Only missing files are queued for the next cell.
expected_times = pd.date_range("2023-01-01", periods=HOUR_LIMIT, freq="h", tz="UTC")

for weather_source, output_h5 in county_h5_by_source.items():
    # 1. Queue a missing file and skip the cache checks.
    if not output_h5.is_file():
        sources_to_download.append(weather_source)
        print(
            f"{weather_source}: no local file; queued for HSDS download in the next code cell.\n  {output_h5.as_posix()}"
        )
        continue

    source_points = selected_source_matches[
        selected_source_matches["weather_source"].eq(weather_source)
    ]

    # 2. Read metadata and identifiers without loading the weather-value arrays.
    with h5py.File(output_h5, "r") as h5_file:
        cached_times = pd.to_datetime(h5_file["time_utc"].asstr()[:], utc=True)
        # County files require all three source metadata attributes to match.
        source_matches = (
            h5_file.attrs.get("kind") == "county_load_weather"
            and h5_file.attrs.get("source") == weather_source
            and h5_file.attrs.get("hsds_resource_path") == SOURCE_RESOURCE_PATHS[weather_source]
        )
        # Check county/grid pairings and their order to preserve column-to-county alignment.
        points_match = np.array_equal(
            h5_file["gids"][:], source_points["selected_gid"].to_numpy(dtype=int)
        ) and np.array_equal(
            h5_file["county_fips"].asstr()[:], source_points["county_fips"].to_numpy()
        )
        # Each required variable must have shape (requested hours, selected counties).
        variables_match = all(
            variable in h5_file and h5_file[variable].shape == (HOUR_LIMIT, len(source_points))
            for variable in SOURCE_VARIABLES[weather_source]
        )
        # Reuse requires every check to pass, including the exact UTC timestamps.
        if not (
            source_matches
            and points_match
            and variables_match
            and cached_times.equals(expected_times)
        ):
            raise ValueError(
                f"Cached weather does not match this example: {output_h5.as_posix()}. Review that file before recreating it with HSDS."
            )

    # 3. Report successful reuse only after every check passes and the file closes.
    print(
        f"{weather_source}: reusing checked local H5 (hours={len(cached_times)}, counties={len(source_points):,}; source metadata verified).\n  {output_h5.as_posix()}"
    )

if not sources_to_download:
    print("All weather files passed the local checks; no HSDS download is needed.")

bchrrr: reusing checked local H5 (hours=1, counties=3,143; source metadata verified).
  ../../data_outputs/data_flow/county_hsds_download_and_ba_weather_aggregation/county_weather_h5/bchrrr_2023_all_county_weather.h5
nsrdb: reusing checked local H5 (hours=1, counties=3,143; source metadata verified).
  ../../data_outputs/data_flow/county_hsds_download_and_ba_weather_aggregation/county_weather_h5/nsrdb_2023_all_county_weather.h5
All weather files passed the local checks; no HSDS download is needed.


### Download missing weather files

Download only queued files using the HSDS settings below. Completed files are removed from the queue, so rerunning this cell skips them. Existing files are never overwritten; if a download leaves a partial file, inspect it before retrying.

In [5]:
HSDS_ENDPOINT = "http://localhost:5101"
HSDS_API_KEY = None
HSDS_BUCKET = "nrel-pds-hsds"
GID_CHUNK_SIZE = 800

if not sources_to_download:
    print("No weather files are queued for download.")

for weather_source in sources_to_download.copy():
    source_points = selected_source_matches[
        selected_source_matches["weather_source"].eq(weather_source)
    ]
    resource_path = SOURCE_RESOURCE_PATHS[weather_source]
    output_h5 = county_h5_by_source[weather_source]
    gids = source_points["selected_gid"].to_numpy(dtype=int)

    print(
        f"{weather_source}: downloading weather for {len(gids):,} counties from HSDS.\n"
        f"  {output_h5.as_posix()}"
    )

    # 1. Open the selected HSDS resource.
    with Resource(
        resource_path,
        hsds=True,
        hsds_kwargs={
            "endpoint": HSDS_ENDPOINT,
            "api_key": HSDS_API_KEY,
            "bucket": HSDS_BUCKET,
        },
    ) as resource:
        # 2. Select hourly timestamps: NSRDB is sub-hourly; BC-HRRR is already hourly.
        time_index = pd.DatetimeIndex(pd.to_datetime(resource.time_index, utc=True))
        selected_times = time_index[time_index.minute == 0]
        selected_times = selected_times[:HOUR_LIMIT]
        time_positions = time_index.get_indexer(selected_times)

        # 3. Create the H5 and write identifiers and metadata.
        with h5py.File(output_h5, "x") as h5_file:
            h5_file.attrs["kind"] = "county_load_weather"
            h5_file.attrs["source"] = weather_source
            h5_file.attrs["hsds_resource_path"] = resource_path
            h5_file.create_dataset(
                "time_utc",
                data=selected_times.strftime("%Y-%m-%dT%H:%M:%SZ").tolist(),
                dtype=h5py.string_dtype("utf-8"),
            )
            h5_file.create_dataset(
                "county_fips",
                data=source_points["county_fips"].tolist(),
                dtype=h5py.string_dtype("utf-8"),
            )
            h5_file.create_dataset("gids", data=gids, dtype="i8")
            h5_file.create_dataset(
                "latitude",
                data=source_points["county_pop_lat"].to_numpy(dtype=float),
                dtype="f8",
            )
            h5_file.create_dataset(
                "longitude",
                data=source_points["county_pop_lon"].to_numpy(dtype=float),
                dtype="f8",
            )

            # 4. Write the weather variables.
            for variable in SOURCE_VARIABLES[weather_source]:
                values = np.empty((len(time_positions), len(gids)), dtype=np.float32)
                for column_start in range(0, len(gids), GID_CHUNK_SIZE):
                    column_end = min(column_start + GID_CHUNK_SIZE, len(gids))
                    gid_chunk = gids[column_start:column_end]
                    for row_index, time_position in enumerate(time_positions):
                        values[row_index, column_start:column_end] = resource[
                            variable, int(time_position), gid_chunk
                        ]
                h5_file.create_dataset(variable, data=values, dtype="f4")

    # 5. Mark the download complete only after the file closes.
    sources_to_download.remove(weather_source)
    print(
        f"{weather_source}: downloaded weather H5 from HSDS "
        f"(hours={len(selected_times)}, counties={len(gids):,}).\n"
        f"  {output_h5.as_posix()}"
    )

No weather files are queued for download.


## Step 3: Calculate BA county weather weights

This step calculates the normalized county weights used to aggregate county weather into one BA-level weather series (MISO is used as an example here). The weight for each county is: `county_weight = county_population / total_population_of_BA_counties`

In [6]:
BA_CODE = "MISO"

ba_mapping = pd.read_csv(
    "../../data_inputs/county_weather/ba_service_territory_2019.csv", dtype=str
)
county_population = pd.read_csv(
    "../../data_inputs/county_weather/county_populations_2000_to_2020.csv",
    usecols=["county_FIPS", "pop_2020"],
    dtype={"county_FIPS": str},
)

# Use one consistent 5-digit county FIPS key in both input tables.
county_fips_parts = ba_mapping["County_FIPS"].str.split(".")
county_fips_text = county_fips_parts.str[0]
ba_mapping["county_fips"] = county_fips_text.str.zfill(5)
county_population["county_fips"] = county_population["county_FIPS"].str.zfill(5)

# Keep the counties assigned to the example BA.
ba_counties = ba_mapping.loc[
    ba_mapping["BA_Code"].eq(BA_CODE),
    ["county_fips", "County_Name", "State_Name"],
].copy()

# Population weights tell Step 4 how much each county contributes to the BA average.
ba_county_weights = ba_counties.merge(
    county_population[["county_fips", "pop_2020"]],
    on="county_fips",
    validate="one_to_one",
)
ba_population = ba_county_weights["pop_2020"].sum()
ba_county_weights["normalized_ba_weather_weight"] = ba_county_weights["pop_2020"] / ba_population

weight_check = pd.DataFrame(
    [
        {
            "BA_Code": BA_CODE,
            "county_count": len(ba_county_weights),
            "sum_normalized_ba_weather_weight": ba_county_weights[
                "normalized_ba_weather_weight"
            ].sum(),
        }
    ]
)

display(
    ba_county_weights.head(5)
    .rename(
        columns={
            "county_fips": "County FIPS",
            "County_Name": "County",
            "State_Name": "State",
            "pop_2020": "2020 population",
            "normalized_ba_weather_weight": "BA weather weight",
        }
    )
    .style.hide(axis="index")
    .format({"2020 population": "{:,.0f}", "BA weather weight": "{:.6f}"})
)
display(
    weight_check.rename(
        columns={
            "BA_Code": "BA",
            "county_count": "Counties",
            "sum_normalized_ba_weather_weight": "Weight sum",
        }
    )
    .style.hide(axis="index")
    .format({"Counties": "{:,.0f}", "Weight sum": "{:.6f}"})
)

County FIPS,County,State,2020 population,BA weather weight
05001,Arkansas County,Arkansas,"17,383",0.000303
05003,Ashley County,Arkansas,"19,339",0.000337
05005,Baxter County,Arkansas,"42,242",0.000737
05009,Boone County,Arkansas,"37,625",0.000657
05011,Bradley County,Arkansas,"10,639",0.000186


BA,Counties,Weight sum
MISO,922,1.000000


## Step 4: Combine county weather into population-weighted BA weather

For each weather variable, the H5 file contains a table with one row per hour and one column per county. This step selects the county columns assigned to `BA_CODE` and weights their values into a single hourly weather series for the BA. Counties with larger populations contribute more to the BA average, using the normalized population weights calculated in Step 3. The calculation is repeated for every weather variable available from each source.

For a given hour, the population-weighted BA value is: `BA weather = sum(county weather value * county population weight)`. The county population weights sum to 1 within the BA.

In [7]:
ba_weather_by_source = {}

for weather_source in WEATHER_SOURCES:
    h5_path = county_h5_by_source[weather_source]
    variables = SOURCE_VARIABLES[weather_source]

    with h5py.File(h5_path, "r") as h5_file:
        time_utc = pd.to_datetime(h5_file["time_utc"].asstr()[:], utc=True)

        # Match H5 county columns to the BA county weights from Step 3.
        h5_counties = pd.DataFrame(
            {
                "county_fips": pd.Series(h5_file["county_fips"].asstr()[:]).str.zfill(5),
                "county_column": np.arange(len(h5_file["county_fips"])),
            }
        )
        ba_county_columns = h5_counties.merge(
            ba_county_weights[["county_fips", "normalized_ba_weather_weight"]],
            on="county_fips",
            validate="one_to_one",
        )

        county_columns = ba_county_columns["county_column"].to_numpy(dtype=int)
        county_weights = ba_county_columns["normalized_ba_weather_weight"].to_numpy(dtype=float)
        assert len(ba_county_columns) == len(ba_county_weights), (
            "Missing BA county weather columns."
        )
        np.testing.assert_allclose(county_weights.sum(), 1.0)

        # For each variable: BA_weather_t = sum(county_weather_i,t * county_weight_i).
        ba_source_weather = pd.DataFrame({"time_utc": time_utc})
        for variable in variables:
            county_weather = h5_file[variable][:, county_columns]
            ba_source_weather[variable] = (county_weather * county_weights).sum(axis=1)

    ba_weather_by_source[weather_source] = ba_source_weather

    print(
        f"{weather_source}: {len(county_columns):,} {BA_CODE} counties, weight sum = {county_weights.sum():.6f}"
    )
    display(
        ba_source_weather.head(5)
        .rename(
            columns={
                "time_utc": "Time (UTC)",
                "temperature_2m": "Temperature at 2 m",
                "specifichumidity_2m": "Specific humidity at 2 m",
                "windspeed_10m": "Wind speed at 10 m",
                "ghi": "GHI",
                "relativehumidity_2m": "Relative humidity at 2 m",
                "pressure_0m": "Surface pressure",
            }
        )
        .style.hide(axis="index")
    )

bchrrr: 922 MISO counties, weight sum = 1.000000


Time (UTC),Temperature at 2 m,Wind speed at 10 m,Relative humidity at 2 m,Surface pressure,Specific humidity at 2 m
2023-01-01 00:00:00+00:00,6.110638,2.562949,84.393737,98880.208226,0.004401


nsrdb: 922 MISO counties, weight sum = 1.000000


Time (UTC),GHI
2023-01-01 00:00:00+00:00,0.000000


## Step 5: Merge BA weather tables and write CSV

BC-HRRR supplies the meteorology variables and NSRDB supplies `ghi`. Join their Step 4 results on the shared hourly `time_utc`, order the native columns, and write the one-hour example CSV. A single-source workflow, such as a sup3rCC model, does not need this cross-source join.

The CSV retains the packaged 2007-2023 BA-weather archive's seven-column schema: `time_utc, temperature_2m, specifichumidity_2m, windspeed_10m, ghi, relativehumidity_2m, pressure_0m`.


In [8]:
ONE_HOUR_BA_WEATHER_CSV = OUTPUT_DIR / "MISO_2023_one_hour_ba_weather.csv"

# BC-HRRR supplies the meteorology variables; NSRDB supplies ghi.
ba_weather_example = ba_weather_by_source["bchrrr"].merge(
    ba_weather_by_source["nsrdb"],
    on="time_utc",
    validate="one_to_one",
)
ba_weather_example = ba_weather_example[["time_utc", *NATIVE_WEATHER_COLUMNS]]

# Format time_utc like the packaged BA-weather CSVs.
ba_weather_example["time_utc"] = ba_weather_example["time_utc"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")

ba_weather_example.to_csv(ONE_HOUR_BA_WEATHER_CSV, index=False)
print(
    f"Wrote one-hour BA-weather CSV ({len(ba_weather_example):,} rows):\n  {ONE_HOUR_BA_WEATHER_CSV.as_posix()}"
)
display(
    ba_weather_example.rename(
        columns={
            "time_utc": "Time (UTC)",
            "temperature_2m": "Temperature at 2 m",
            "specifichumidity_2m": "Specific humidity at 2 m",
            "windspeed_10m": "Wind speed at 10 m",
            "ghi": "GHI",
            "relativehumidity_2m": "Relative humidity at 2 m",
            "pressure_0m": "Surface pressure",
        }
    ).style.hide(axis="index")
)

Wrote one-hour BA-weather CSV (1 rows):
  ../../data_outputs/data_flow/county_hsds_download_and_ba_weather_aggregation/MISO_2023_one_hour_ba_weather.csv


Time (UTC),Temperature at 2 m,Specific humidity at 2 m,Wind speed at 10 m,GHI,Relative humidity at 2 m,Surface pressure
2023-01-01T00:00:00Z,6.110638,0.004401,2.562949,0.000000,84.393737,98880.208226


## Interpretation

The CSV is a one-hour demonstration of population-weighted MISO weather, using fixed 2020 county populations. It is not the complete historical weather input used by TELL. The source tables must retain their county order, and county weights must sum to one within the BA.

Appendix A documents WTK specific humidity; it is reference-only for the BC-HRRR/NSRDB example above. Appendix B applies the same weighted-average calculation to Iowa counties.


## Appendix A: WTK specific humidity derivation

WTK does not provide `specifichumidity_2m`, so WTK workflows derive it while writing the county-weather H5 file in Step 2. The derivation happens at the county-hour level before BA aggregation:

1. Read WTK `temperature_2m`, `relativehumidity_2m`, and `pressure_0m`.
2. Calculate `specifichumidity_2m` for each county and hour.
3. Write the derived `specifichumidity_2m` dataset into the WTK county-weather H5.
4. Use the same Step 4 population-weighting method as the other variables.

The specific humidity equations come from: [CEOP derived-parameter equations](https://www.cen.uni-hamburg.de/en/icdc/data/atmosphere/docs-atmo/ceop-derived-parameter-equations.pdf)

In [9]:
temperature_2m_c = 20
relativehumidity_2m_pct = 100
pressure_0m_pa = 101325

# Saturation vapor pressure over liquid water, in hPa.
e_s = 6.112 * np.exp((17.67 * temperature_2m_c) / (temperature_2m_c + 243.5))

# Actual vapor pressure from relative humidity.
e = e_s * relativehumidity_2m_pct / 100.0

# Specific humidity in kg kg-1.
specifichumidity_2m = (0.622 * e) / (pressure_0m_pa / 100.0 - 0.378 * e)
print(f"Specific humidity: {specifichumidity_2m:.6f} kg/kg")

Specific humidity: 0.014472 kg/kg


## Appendix B: State county-population weather aggregation

This process aggregates county-level weather data to the state-level (instead of BA-level), using Iowa as an example. It is analogous to Step 4, but using all the counties in Iowa instead of all the counties in MISO.

In [10]:
STATE_NAME = "Iowa"

# Select one unique row for every Iowa county.
iowa_counties = regrid_matches.loc[regrid_matches["state_name"].eq(STATE_NAME), ["county_fips"]]
iowa_counties = iowa_counties.drop_duplicates()
iowa_county_weights = iowa_counties.merge(
    county_population[["county_fips", "pop_2020"]],
    on="county_fips",
    validate="one_to_one",
)

# Normalize fixed 2020 population within Iowa.
iowa_population = iowa_county_weights["pop_2020"].sum()
iowa_county_weights["normalized_state_weather_weight"] = (
    iowa_county_weights["pop_2020"] / iowa_population
)
state_weather_by_source = {}

for weather_source in WEATHER_SOURCES:
    with h5py.File(county_h5_by_source[weather_source], "r") as h5_file:
        h5_counties = pd.DataFrame(
            {
                "county_fips": pd.Series(h5_file["county_fips"].asstr()[:]).str.zfill(5),
                "county_column": np.arange(len(h5_file["county_fips"])),
            }
        )
        state_columns = h5_counties.merge(
            iowa_county_weights, on="county_fips", validate="one_to_one"
        )
        columns = state_columns["county_column"].to_numpy(dtype=int)
        weights = state_columns["normalized_state_weather_weight"].to_numpy(dtype=float)
        time_utc = pd.to_datetime(h5_file["time_utc"].asstr()[:], utc=True)
        state_source_weather = pd.DataFrame({"time_utc": time_utc})
        for variable in SOURCE_VARIABLES[weather_source]:
            county_weather = h5_file[variable][:, columns]
            state_source_weather[variable] = (county_weather * weights).sum(axis=1)
    state_weather_by_source[weather_source] = state_source_weather

# Merge the source-specific results into the native seven-column schema.
iowa_weather_example = state_weather_by_source["bchrrr"].merge(
    state_weather_by_source["nsrdb"],
    on="time_utc",
    validate="one_to_one",
)
iowa_weather_example = iowa_weather_example[["time_utc", *NATIVE_WEATHER_COLUMNS]]
display(
    pd.DataFrame(
        {
            "State": [STATE_NAME],
            "Counties": [len(iowa_county_weights)],
            "Weight sum": [iowa_county_weights["normalized_state_weather_weight"].sum()],
        }
    )
    .style.hide(axis="index")
    .format({"Counties": "{:,.0f}", "Weight sum": "{:.6f}"})
)
display(
    iowa_weather_example.head(1)
    .rename(
        columns={
            "time_utc": "Time (UTC)",
            "temperature_2m": "Temperature at 2 m",
            "specifichumidity_2m": "Specific humidity at 2 m",
            "windspeed_10m": "Wind speed at 10 m",
            "ghi": "GHI",
            "relativehumidity_2m": "Relative humidity at 2 m",
            "pressure_0m": "Surface pressure",
        }
    )
    .style.hide(axis="index")
)

State,Counties,Weight sum
Iowa,99,1.000000


Time (UTC),Temperature at 2 m,Specific humidity at 2 m,Wind speed at 10 m,GHI,Relative humidity at 2 m,Surface pressure
2023-01-01 00:00:00+00:00,1.446505,0.000102,3.177773,0.000000,88.024819,97415.611246
